In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
!pip install faster-whisper jiwer speechbrain librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 54.6 MB/s eta 0:00:00


In [1]:
# @title
from pathlib import Path
from tqdm import tqdm

import librosa
import torch
import pandas as pd

from speechbrain.inference.speaker import EncoderClassifier


# ==========================================
# Load ECAPA once
# ==========================================

speaker_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": "cuda"}
)


# ==========================================
# Extract embedding
# ==========================================

def get_embedding(audio_path):

    audio, _ = librosa.load(
        audio_path,
        sr=16000,
        mono=True
    )

    audio = torch.tensor(
        audio,
        dtype=torch.float32
    ).unsqueeze(0)

    with torch.no_grad():

        emb = speaker_model.encode_batch(
            audio
        )

    return emb.squeeze()


# ==========================================
# Folder evaluation
# ==========================================

def evaluate_speaker_similarity(
    reference_audio,
    fake_dir,
    output_csv
):

    fake_dir = Path(fake_dir)

    fake_files = sorted(
        fake_dir.glob("*.wav")
    )

    print(
        f"Found {len(fake_files)} generated files"
    )

    # ----------------------------------
    # Reference embedding computed ONCE
    # ----------------------------------

    print("Computing reference embedding...")

    ref_emb = get_embedding(
        reference_audio
    )

    results = []

    for fake_file in tqdm(
        fake_files,
        desc="Speaker Similarity"
    ):

        fake_emb = get_embedding(
            fake_file
        )

        similarity = torch.nn.functional.cosine_similarity(
            ref_emb,
            fake_emb,
            dim=0
        )

        results.append({

            "filename":
                fake_file.name,

            "speaker_similarity":
                float(similarity)

        })

    df = pd.DataFrame(
        results
    )

    df.to_csv(
        output_csv,
        index=False
    )

    print(
        f"\nSaved CSV:\n{output_csv}"
    )

    print("\n" + "=" * 80)
    print("SPEAKER SIMILARITY SUMMARY")
    print("=" * 80)

    print(
        f"Files processed: {len(df)}"
    )

    print(
        f"\nMean   : {df['speaker_similarity'].mean():.4f}"
    )

    print(
        f"Median : {df['speaker_similarity'].median():.4f}"
    )

    print(
        f"Std    : {df['speaker_similarity'].std():.4f}"
    )

    print(
        f"Min    : {df['speaker_similarity'].min():.4f}"
    )

    print(
        f"Max    : {df['speaker_similarity'].max():.4f}"
    )

    print("\n========== WORST 10 ==========\n")

    print(
        df.sort_values(
            "speaker_similarity"
        ).head(10)
    )

    return df

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


In [5]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_neutral_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_jennifer/audio",
    output_csv = "jen_netraul_vs_jen_vc_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:17<00:00, 22.82it/s]


Saved CSV:
jen_netraul_vs_jen_vc_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.5529
Median : 0.5596
Std    : 0.0519
Min    : 0.3459
Max    : 0.6846

========== WORST 10 ==========

                          filename  speaker_similarity
111    5322_7680_000061_000006.wav            0.345859
80     5322_7679_000025_000005.wav            0.370424
133   6209_34600_000007_000002.wav            0.388421
26    250_142286_000006_000000.wav            0.390689
148   6209_34600_000027_000001.wav            0.392991
382  8312_279791_000022_000000.wav            0.397764
102    5322_7680_000048_000000.wav            0.400209
13    250_142276_000006_000004.wav            0.418516
198   6209_34601_000120_000000.wav            0.420605
125   6209_34599_000020_000001.wav            0.426188


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.642066
1,250_140277_000004_000001.wav,0.581718
2,250_140277_000004_000002.wav,0.587600
3,250_140277_000005_000000.wav,0.574412
4,250_142276_000003_000002.wav,0.621392
...,...,...
395,8312_279791_000051_000000.wav,0.538829
396,8312_279791_000051_000001.wav,0.613572
397,8312_279791_000053_000001.wav,0.609342
398,8312_279791_000053_000004.wav,0.480857


In [2]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_neutral_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_sad/audio",
    output_csv = "jen_netraul_vs_jen_evc_sad_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:18<00:00, 21.60it/s]



Saved CSV:
jen_netraul_vs_jen_evc_sad_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.3914
Median : 0.4001
Std    : 0.0511
Min    : 0.2061
Max    : 0.5138

========== WORST 10 ==========

                          filename  speaker_similarity
79     5322_7679_000024_000000.wav            0.206146
26    250_142286_000006_000000.wav            0.228342
163   6209_34601_000052_000002.wav            0.229801
199   6209_34601_000134_000000.wav            0.237821
239  7800_283492_000016_000001.wav            0.240393
297       78_369_000008_000003.wav            0.252975
177   6209_34601_000084_000001.wav            0.260416
102    5322_7680_000048_000000.wav            0.267914
356  8312_279790_000026_000000.wav            0.268725
204   6209_34601_000156_000001.wav            0.273483


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.424444
1,250_140277_000004_000001.wav,0.387389
2,250_140277_000004_000002.wav,0.349675
3,250_140277_000005_000000.wav,0.423805
4,250_142276_000003_000002.wav,0.415991
...,...,...
395,8312_279791_000051_000000.wav,0.354402
396,8312_279791_000051_000001.wav,0.430993
397,8312_279791_000053_000001.wav,0.426289
398,8312_279791_000053_000004.wav,0.413891


In [3]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_sad_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_sad/audio",
    output_csv = "jen_sad_vs_jen_evc_sad_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:09<00:00, 42.24it/s]


Saved CSV:
jen_sad_vs_jen_evc_sad_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.6196
Median : 0.6291
Std    : 0.0552
Min    : 0.3576
Max    : 0.7295

========== WORST 10 ==========

                          filename  speaker_similarity
106    5322_7680_000057_000000.wav            0.357550
163   6209_34601_000052_000002.wav            0.386352
80     5322_7679_000025_000005.wav            0.415444
46    250_142286_000053_000000.wav            0.447840
66     5322_7679_000002_000003.wav            0.448366
256  7800_283493_000006_000000.wav            0.465903
34    250_142286_000024_000001.wav            0.472695
263  7800_283493_000018_000000.wav            0.488326
36    250_142286_000028_000000.wav            0.488804
13    250_142276_000006_000004.wav            0.489930


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.691866
1,250_140277_000004_000001.wav,0.663655
2,250_140277_000004_000002.wav,0.590460
3,250_140277_000005_000000.wav,0.663376
4,250_142276_000003_000002.wav,0.641764
...,...,...
395,8312_279791_000051_000000.wav,0.597827
396,8312_279791_000051_000001.wav,0.640551
397,8312_279791_000053_000001.wav,0.665681
398,8312_279791_000053_000004.wav,0.604266


In [4]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_neutral_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_angry/audio",
    output_csv = "jen_netraul_vs_jen_evc_angry_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:17<00:00, 23.19it/s]


Saved CSV:
jen_netraul_vs_jen_evc_angry_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.2942
Median : 0.2929
Std    : 0.0532
Min    : 0.0843
Max    : 0.4298

========== WORST 10 ==========

                          filename  speaker_similarity
256  7800_283493_000006_000000.wav            0.084329
138   6209_34600_000009_000002.wav            0.140392
120   6209_34599_000009_000000.wav            0.155052
259  7800_283493_000011_000002.wav            0.155355
302       78_369_000013_000006.wav            0.166956
307       78_369_000020_000001.wav            0.172221
88     5322_7680_000014_000000.wav            0.172423
261  7800_283493_000013_000001.wav            0.175754
155   6209_34601_000040_000001.wav            0.177751
156   6209_34601_000041_000000.wav            0.182849


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.332014
1,250_140277_000004_000001.wav,0.322703
2,250_140277_000004_000002.wav,0.228640
3,250_140277_000005_000000.wav,0.294987
4,250_142276_000003_000002.wav,0.311311
...,...,...
395,8312_279791_000051_000000.wav,0.215346
396,8312_279791_000051_000001.wav,0.309077
397,8312_279791_000053_000001.wav,0.250517
398,8312_279791_000053_000004.wav,0.369447


In [ ]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_angry_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_angry/audio",
    output_csv = "jen_angry_vs_jen_evc_angry_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:09<00:00, 40.36it/s]


Saved CSV:
jen_angry_vs_jen_evc_angry_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.5579
Median : 0.5672
Std    : 0.0635
Min    : 0.3091
Max    : 0.7160

========== WORST 10 ==========

                          filename  speaker_similarity
256  7800_283493_000006_000000.wav            0.309134
124   6209_34599_000018_000003.wav            0.315967
297       78_369_000008_000003.wav            0.375355
278  7800_283493_000067_000001.wav            0.392491
302       78_369_000013_000006.wav            0.400377
235  7800_283492_000010_000001.wav            0.410613
177   6209_34601_000084_000001.wav            0.412472
201   6209_34601_000151_000000.wav            0.415612
119   6209_34599_000008_000005.wav            0.416032
348  8312_279790_000012_000000.wav            0.417335


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.652523
1,250_140277_000004_000001.wav,0.635895
2,250_140277_000004_000002.wav,0.579524
3,250_140277_000005_000000.wav,0.628015
4,250_142276_000003_000002.wav,0.608290
...,...,...
395,8312_279791_000051_000000.wav,0.586007
396,8312_279791_000051_000001.wav,0.580732
397,8312_279791_000053_000001.wav,0.563382
398,8312_279791_000053_000004.wav,0.580631


In [ ]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_neutral_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_david/audio",
    output_csv = "david_netraul_vs_david_vc_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:09<00:00, 40.00it/s]


Saved CSV:
david_netraul_vs_david_vc_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.6396
Median : 0.6485
Std    : 0.0596
Min    : 0.4558
Max    : 0.7628

========== WORST 10 ==========

                          filename  speaker_similarity
278  7800_283493_000067_000001.wav            0.455845
158   6209_34601_000042_000002.wav            0.458299
295       78_368_000021_000001.wav            0.473023
184   6209_34601_000096_000044.wav            0.476599
238  7800_283492_000016_000000.wav            0.479655
261  7800_283493_000013_000001.wav            0.482294
112    5322_7680_000061_000007.wav            0.494365
281  7800_283493_000071_000001.wav            0.502506
147   6209_34600_000026_000001.wav            0.502797
282  7800_283493_000072_000001.wav            0.506710


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.715567
1,250_140277_000004_000001.wav,0.647832
2,250_140277_000004_000002.wav,0.654382
3,250_140277_000005_000000.wav,0.673973
4,250_142276_000003_000002.wav,0.659077
...,...,...
395,8312_279791_000051_000000.wav,0.598795
396,8312_279791_000051_000001.wav,0.651093
397,8312_279791_000053_000001.wav,0.705619
398,8312_279791_000053_000004.wav,0.671898


In [ ]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_neutral_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_sad/audio",
    output_csv = "david_netraul_vs_david_evc_sad_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:12<00:00, 32.58it/s]


Saved CSV:
david_netraul_vs_david_evc_sad_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.4514
Median : 0.4568
Std    : 0.0569
Min    : 0.2685
Max    : 0.6250

========== WORST 10 ==========

                          filename  speaker_similarity
116   6209_34599_000003_000000.wav            0.268495
106    5322_7680_000057_000000.wav            0.276893
115    5322_7680_000064_000000.wav            0.285474
348  8312_279790_000012_000000.wav            0.292606
57     5322_7678_000006_000008.wav            0.293016
158   6209_34601_000042_000002.wav            0.294462
119   6209_34599_000008_000005.wav            0.294879
251  7800_283492_000049_000001.wav            0.297446
176   6209_34601_000084_000000.wav            0.313714
295       78_368_000021_000001.wav            0.315837


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.429931
1,250_140277_000004_000001.wav,0.477760
2,250_140277_000004_000002.wav,0.426970
3,250_140277_000005_000000.wav,0.469469
4,250_142276_000003_000002.wav,0.492919
...,...,...
395,8312_279791_000051_000000.wav,0.390888
396,8312_279791_000051_000001.wav,0.536008
397,8312_279791_000053_000001.wav,0.399596
398,8312_279791_000053_000004.wav,0.445844


In [ ]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_sad_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_sad/audio",
    output_csv = "david_sad_vs_david_evc_sad_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:11<00:00, 35.83it/s]


Saved CSV:
david_sad_vs_david_evc_sad_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.6095
Median : 0.6185
Std    : 0.0583
Min    : 0.4045
Max    : 0.7308

========== WORST 10 ==========

                          filename  speaker_similarity
116   6209_34599_000003_000000.wav            0.404544
106    5322_7680_000057_000000.wav            0.419427
318       78_369_000043_000008.wav            0.435846
176   6209_34601_000084_000000.wav            0.440677
57     5322_7678_000006_000008.wav            0.445106
158   6209_34601_000042_000002.wav            0.453081
88     5322_7680_000014_000000.wav            0.454048
348  8312_279790_000012_000000.wav            0.462537
170   6209_34601_000068_000009.wav            0.470396
115    5322_7680_000064_000000.wav            0.470624


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.597145
1,250_140277_000004_000001.wav,0.629319
2,250_140277_000004_000002.wav,0.557203
3,250_140277_000005_000000.wav,0.578034
4,250_142276_000003_000002.wav,0.639388
...,...,...
395,8312_279791_000051_000000.wav,0.534935
396,8312_279791_000051_000001.wav,0.679823
397,8312_279791_000053_000001.wav,0.593247
398,8312_279791_000053_000004.wav,0.637500


In [ ]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_neutral_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_angry/audio",
    output_csv = "david_netraul_vs_david_evc_angry_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:11<00:00, 33.75it/s]



Saved CSV:
david_netraul_vs_david_evc_angry_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.2418
Median : 0.2486
Std    : 0.0770
Min    : -0.1598
Max    : 0.4341

========== WORST 10 ==========

                          filename  speaker_similarity
239  7800_283492_000016_000001.wav           -0.159815
275  7800_283493_000060_000000.wav           -0.088761
269  7800_283493_000037_000000.wav            0.026821
102    5322_7680_000048_000000.wav            0.064556
207  7800_283478_000011_000001.wav            0.071193
251  7800_283492_000049_000001.wav            0.073698
352  8312_279790_000019_000004.wav            0.075280
158   6209_34601_000042_000002.wav            0.077151
209  7800_283478_000015_000001.wav            0.077792
345  8312_279790_000008_000000.wav            0.082086


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.236771
1,250_140277_000004_000001.wav,0.222976
2,250_140277_000004_000002.wav,0.194859
3,250_140277_000005_000000.wav,0.173362
4,250_142276_000003_000002.wav,0.304433
...,...,...
395,8312_279791_000051_000000.wav,0.175230
396,8312_279791_000051_000001.wav,0.291869
397,8312_279791_000053_000001.wav,0.132287
398,8312_279791_000053_000004.wav,0.299389


In [ ]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_angry_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_angry/audio",
    output_csv = "david_angry_vs_david_evc_angry_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:11<00:00, 35.17it/s]



Saved CSV:
david_angry_vs_david_evc_angry_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.4860
Median : 0.4931
Std    : 0.0844
Min    : -0.1592
Max    : 0.6455

========== WORST 10 ==========

                          filename  speaker_similarity
239  7800_283492_000016_000001.wav           -0.159178
275  7800_283493_000060_000000.wav           -0.084049
102    5322_7680_000048_000000.wav            0.261416
251  7800_283492_000049_000001.wav            0.280732
158   6209_34601_000042_000002.wav            0.282188
269  7800_283493_000037_000000.wav            0.283397
352  8312_279790_000019_000004.wav            0.313339
209  7800_283478_000015_000001.wav            0.313386
119   6209_34599_000008_000005.wav            0.324540
150   6209_34600_000029_000006.wav            0.325294


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.462373
1,250_140277_000004_000001.wav,0.488728
2,250_140277_000004_000002.wav,0.462410
3,250_140277_000005_000000.wav,0.354984
4,250_142276_000003_000002.wav,0.528773
...,...,...
395,8312_279791_000051_000000.wav,0.506101
396,8312_279791_000051_000001.wav,0.570741
397,8312_279791_000053_000001.wav,0.384901
398,8312_279791_000053_000004.wav,0.567882


In [ ]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_neutral_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_morgan/audio",
    output_csv = "morgan_neutral_vs_morgan_vc_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:11<00:00, 36.31it/s]


Saved CSV:
morgan_neutral_vs_morgan_vc_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.5891
Median : 0.5954
Std    : 0.0561
Min    : 0.3852
Max    : 0.7252

========== WORST 10 ==========

                          filename  speaker_similarity
150   6209_34600_000029_000006.wav            0.385219
158   6209_34601_000042_000002.wav            0.390412
115    5322_7680_000064_000000.wav            0.416265
263  7800_283493_000018_000000.wav            0.442738
175   6209_34601_000080_000001.wav            0.449273
20    250_142276_000014_000002.wav            0.459748
125   6209_34599_000020_000001.wav            0.460361
83     5322_7679_000029_000000.wav            0.468113
356  8312_279790_000026_000000.wav            0.468766
210  7800_283478_000018_000002.wav            0.471989


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.617979
1,250_140277_000004_000001.wav,0.637784
2,250_140277_000004_000002.wav,0.629482
3,250_140277_000005_000000.wav,0.619430
4,250_142276_000003_000002.wav,0.659421
...,...,...
395,8312_279791_000051_000000.wav,0.552747
396,8312_279791_000051_000001.wav,0.615202
397,8312_279791_000053_000001.wav,0.630268
398,8312_279791_000053_000004.wav,0.549065


In [ ]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_neutral_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_sad/audio",
    output_csv = "morgan_neutral_vs_morgan_evc_sad_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:11<00:00, 33.39it/s]


Saved CSV:
morgan_neutral_vs_morgan_evc_sad_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.4156
Median : 0.4301
Std    : 0.1021
Min    : -0.1152
Max    : 0.6181

========== WORST 10 ==========

                         filename  speaker_similarity
175  6209_34601_000080_000001.wav           -0.115173
158  6209_34601_000042_000002.wav           -0.113297
198  6209_34601_000120_000000.wav           -0.097133
91    5322_7680_000019_000001.wav           -0.095936
166  6209_34601_000065_000001.wav           -0.095160
35   250_142286_000025_000000.wav           -0.094280
94    5322_7680_000023_000000.wav           -0.030067
203  6209_34601_000154_000000.wav           -0.023972
204  6209_34601_000156_000001.wav           -0.015440
74    5322_7679_000018_000003.wav            0.033065


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.471750
1,250_140277_000004_000001.wav,0.387831
2,250_140277_000004_000002.wav,0.383868
3,250_140277_000005_000000.wav,0.368877
4,250_142276_000003_000002.wav,0.370225
...,...,...
395,8312_279791_000051_000000.wav,0.509924
396,8312_279791_000051_000001.wav,0.427473
397,8312_279791_000053_000001.wav,0.504115
398,8312_279791_000053_000004.wav,0.417525


In [ ]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_sad_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_sad/audio",
    output_csv = "morgan_sad_vs_morgan_evc_sad_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:09<00:00, 41.90it/s]


Saved CSV:
morgan_sad_vs_morgan_evc_sad_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.5877
Median : 0.6307
Std    : 0.1472
Min    : -0.1090
Max    : 0.7433

========== WORST 10 ==========

                         filename  speaker_similarity
175  6209_34601_000080_000001.wav           -0.108972
91    5322_7680_000019_000001.wav           -0.108128
158  6209_34601_000042_000002.wav           -0.102968
166  6209_34601_000065_000001.wav           -0.095166
35   250_142286_000025_000000.wav           -0.093847
198  6209_34601_000120_000000.wav           -0.089724
203  6209_34601_000154_000000.wav           -0.085554
94    5322_7680_000023_000000.wav           -0.023154
204  6209_34601_000156_000001.wav            0.000532
62    5322_7678_000006_000026.wav            0.017209


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.648403
1,250_140277_000004_000001.wav,0.620140
2,250_140277_000004_000002.wav,0.669268
3,250_140277_000005_000000.wav,0.612385
4,250_142276_000003_000002.wav,0.601357
...,...,...
395,8312_279791_000051_000000.wav,0.665690
396,8312_279791_000051_000001.wav,0.664289
397,8312_279791_000053_000001.wav,0.612327
398,8312_279791_000053_000004.wav,0.626530


In [ ]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_neutral_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_angry/audio",
    output_csv = "morgan_neutral_vs_morgan_evc_angry_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:11<00:00, 36.21it/s]


Saved CSV:
morgan_neutral_vs_morgan_evc_angry_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.3752
Median : 0.3819
Std    : 0.0627
Min    : -0.0649
Max    : 0.5267

========== WORST 10 ==========

                          filename  speaker_similarity
166   6209_34601_000065_000001.wav           -0.064887
287       78_368_000010_000002.wav            0.161289
123   6209_34599_000018_000002.wav            0.179486
321       78_369_000048_000001.wav            0.209458
183   6209_34601_000096_000033.wav            0.216458
38    250_142286_000030_000004.wav            0.218239
48    250_142286_000055_000001.wav            0.221839
186   6209_34601_000096_000057.wav            0.224933
387  8312_279791_000036_000000.wav            0.228024
221  7800_283478_000036_000001.wav            0.229012


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.390235
1,250_140277_000004_000001.wav,0.418686
2,250_140277_000004_000002.wav,0.379179
3,250_140277_000005_000000.wav,0.407461
4,250_142276_000003_000002.wav,0.310383
...,...,...
395,8312_279791_000051_000000.wav,0.344415
396,8312_279791_000051_000001.wav,0.289503
397,8312_279791_000053_000001.wav,0.304585
398,8312_279791_000053_000004.wav,0.460029


In [ ]:
# @title
evaluate_speaker_similarity(
    reference_audio = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_angry_ref.wav",
    fake_dir = "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_angry/audio",
    output_csv = "morgan_angry_vs_morgan_evc_angry_SpeakerSimilarirty"
)

Found 400 generated files
Computing reference embedding...


Speaker Similarity: 100%|██████████| 400/400 [00:10<00:00, 39.41it/s]



Saved CSV:
morgan_angry_vs_morgan_evc_angry_SpeakerSimilarirty

SPEAKER SIMILARITY SUMMARY
Files processed: 400

Mean   : 0.6310
Median : 0.6447
Std    : 0.0815
Min    : 0.0184
Max    : 0.7726

========== WORST 10 ==========

                          filename  speaker_similarity
166   6209_34601_000065_000001.wav            0.018447
66     5322_7679_000002_000003.wav            0.363395
80     5322_7679_000025_000005.wav            0.372997
133   6209_34600_000007_000002.wav            0.381043
158   6209_34601_000042_000002.wav            0.391724
106    5322_7680_000057_000000.wav            0.394947
382  8312_279791_000022_000000.wav            0.400752
154   6209_34601_000025_000000.wav            0.402987
209  7800_283478_000015_000001.wav            0.417083
367  8312_279790_000045_000001.wav            0.429984


,filename,speaker_similarity
0,250_140277_000004_000000.wav,0.682562
1,250_140277_000004_000001.wav,0.677726
2,250_140277_000004_000002.wav,0.704556
3,250_140277_000005_000000.wav,0.644685
4,250_142276_000003_000002.wav,0.583492
...,...,...
395,8312_279791_000051_000000.wav,0.675612
396,8312_279791_000051_000001.wav,0.557805
397,8312_279791_000053_000001.wav,0.679131
398,8312_279791_000053_000004.wav,0.615178


In [21]:
from faster_whisper import WhisperModel
from jiwer import wer, cer
from pathlib import Path
from tqdm import tqdm

import pandas as pd
import re
import string
import numpy as np
import random
import torch

# ==================================================
# DETERMINISM
# ==================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==================================================
# TEXT NORMALIZATION
# ==================================================

def normalize_text(text):

    text = text.lower()

    text = text.replace("--", " ")
    text = text.replace("—", " ")
    text = text.replace("–", " ")
    text = text.replace("-", " ")

    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

# ==================================================
# LOAD WHISPER
# ==================================================

print("Loading Whisper Large-v3...")

asr_model = WhisperModel(
    "large-v3",
    device="cuda",
    compute_type="float16"
)

print("Whisper loaded.")

# ==================================================
# TRANSCRIBE
# ==================================================

def transcribe_audio(audio_path):

    segments, _ = asr_model.transcribe(
        str(audio_path),
        language="en",
        beam_size=5
    )

    return " ".join(
        seg.text.strip()
        for seg in segments
    ).strip()

# ==================================================
# GET REFERENCE TEXT
# ==================================================

def get_reference_text(
    stem,
    reference_dir
):

    txt_file = (
        reference_dir /
        f"{stem}.txt"
    )

    if txt_file.exists():

        with open(
            txt_file,
            "r",
            encoding="utf-8"
        ) as f:

            return f.read().strip()

    wav_file = (
        reference_dir /
        f"{stem}.wav"
    )

    if wav_file.exists():

        return transcribe_audio(
            wav_file
        )

    return None

# ==================================================
# MAIN FUNCTION
# ==================================================

def evaluate_asr(
    reference_dir,
    audio_dir,
    output_csv
):

    reference_dir = Path(
        reference_dir
    )

    audio_dir = Path(
        audio_dir
    )

    audio_files = sorted(
        audio_dir.glob("*.wav")
    )

    print(
        f"\nFound {len(audio_files)} files"
    )

    results = []

    for audio_file in tqdm(
        audio_files,
        desc="Processing"
    ):

        gt_text = get_reference_text(
            audio_file.stem,
            reference_dir
        )

        if gt_text is None:

            print(
                f"Missing reference for "
                f"{audio_file.name}"
            )

            continue

        pred_text = transcribe_audio(
            audio_file
        )

        gt_norm = normalize_text(
            gt_text
        )

        pred_norm = normalize_text(
            pred_text
        )

        sample_wer = wer(
            gt_norm,
            pred_norm
        )

        sample_cer = cer(
            gt_norm,
            pred_norm
        )

        results.append({

            "filename":
                audio_file.name,

            "ground_truth":
                gt_text,

            "prediction":
                pred_text,

            "ground_truth_normalized":
                gt_norm,

            "prediction_normalized":
                pred_norm,

            "wer":
                sample_wer,

            "cer":
                sample_cer
        })

    # ==================================================
    # SAVE CSV
    # ==================================================

    df = pd.DataFrame(
        results
    )

    df.to_csv(
        output_csv,
        index=False
    )

    print(
        f"\nSaved CSV:\n{output_csv}"
    )

    # ==================================================
    # SUMMARY
    # ==================================================

    print("\n")
    print("=" * 80)
    print("SUMMARY")
    print("=" * 80)

    print(
        f"Files processed: "
        f"{len(df)}"
    )

    print("\n----- WER -----")

    print(
        f"Mean   : "
        f"{df['wer'].mean():.4f}"
    )

    print(
        f"Median : "
        f"{df['wer'].median():.4f}"
    )

    print(
        f"Std    : "
        f"{df['wer'].std():.4f}"
    )

    print(
        f"Min    : "
        f"{df['wer'].min():.4f}"
    )

    print(
        f"Max    : "
        f"{df['wer'].max():.4f}"
    )

    print("\n----- CER -----")

    print(
        f"Mean   : "
        f"{df['cer'].mean():.4f}"
    )

    print(
        f"Median : "
        f"{df['cer'].median():.4f}"
    )

    print(
        f"Std    : "
        f"{df['cer'].std():.4f}"
    )

    print(
        f"Min    : "
        f"{df['cer'].min():.4f}"
    )

    print(
        f"Max    : "
        f"{df['cer'].max():.4f}"
    )

    print("\n----- QUALITY -----")

    print(
        f"Perfect WER: "
        f"{(df['wer'] == 0).mean() * 100:.2f}%"
    )

    print(
        f"WER <= 0.10: "
        f"{(df['wer'] <= 0.10).mean() * 100:.2f}%"
    )

    print(
        f"WER <= 0.20: "
        f"{(df['wer'] <= 0.20).mean() * 100:.2f}%"
    )

    print(
        f"CER <= 0.10: "
        f"{(df['cer'] <= 0.10).mean() * 100:.2f}%"
    )

    # ==================================================
    # WORST SAMPLES
    # ==================================================

    print(
        "\n========== TOP 10 WORST WER ==========\n"
    )

    worst = df.sort_values(
        "wer",
        ascending=False
    )

    for _, row in worst.head(10).iterrows():

        print("=" * 80)

        print(
            row["filename"]
        )

        print(
            f"WER: {row['wer']:.4f}"
        )

        print(
            f"CER: {row['cer']:.4f}"
        )

        print("\nGROUND TRUTH:")
        print(
            row["ground_truth"]
        )

        print("\nPREDICTION:")
        print(
            row["prediction"]
        )

    return df

Loading Whisper Large-v3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Whisper loaded.


In [22]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_jennifer/audio",
    output_csv="/content/vc_jennifer_results.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [09:30<00:00,  1.43s/it]


Saved CSV:
/content/vc_jennifer_results.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.0393
Median : 0.0000
Std    : 0.1103
Min    : 0.0000
Max    : 1.2647

----- CER -----
Mean   : 0.0173
Median : 0.0000
Std    : 0.0675
Min    : 0.0000
Max    : 1.0549

----- QUALITY -----
Perfect WER: 68.50%
WER <= 0.10: 88.50%
WER <= 0.20: 95.50%
CER <= 0.10: 97.00%

========== TOP 10 WORST WER ==========

6209_34601_000058_000000.wav
WER: 1.2647
CER: 1.0549

GROUND TRUTH:
The man kicked the stool forward and made the little boy sit down, again shoving him by the shoulders; then he pointed with his finger to the porringer which was smoking upon the stove.

PREDICTION:
the man kicked the stool forward and made the little boy sit down again shoving him by the shoulders then he pointed with his finger to the porringer which was smoking upon the stove he said to a boy how did you ever come this far? the boy said that if you were alone, you shouldaltrawn however the boy said he would come

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.000000,0.000000
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,for the same reason no attention has been give...,0.000000,0.000000
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,0.000000,0.000000
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",part of chapter four has already appeared in t...,part of chapter iv has already appeared in the...,part of chapter four has already appeared in t...,0.040816,0.025078
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,0.028571,0.010526
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","Does he so? said the queen, now comprehending ...",does he so said the queen now comprehending all,does he so said the queen now comprehending all,0.000000,0.000000
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",than if you will promise to-night to keep the ...,then if you will promise to night to keep the ...,than if you will promise to night to keep the ...,0.029412,0.006024
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.000000,0.000000
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [23]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_angry/audio",
    output_csv="evc_jennifer_angry_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [06:57<00:00,  1.04s/it]


Saved CSV:
evc_jennifer_angry_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.1176
Median : 0.0769
Std    : 0.1518
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0600
Median : 0.0359
Std    : 0.0832
Min    : 0.0000
Max    : 0.6538

----- QUALITY -----
Perfect WER: 28.50%
WER <= 0.10: 61.00%
WER <= 0.20: 83.50%
CER <= 0.10: 81.50%

========== TOP 10 WORST WER ==========

6209_34599_000018_000002.wav
WER: 1.0000
CER: 0.3478

GROUND TRUTH:
Roofs--dwellings--shelter!

PREDICTION:
It proves dueling's shelter.
5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.6538

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
airhomburg.locasca
5322_7680_000028_000001.wav
WER: 0.8571
CER: 0.4348

GROUND TRUTH:
Bigger than ours?' asked Lukashka good-naturedly.

PREDICTION:
a bicker than ours, ass-lacash-cat-ganitrably.
5322_7679_000024_000000.wav
WER: 0.8571
CER: 0.3889

GROUND TRUTH:
'It's just a habit,' answered Olenin. 'Why?'

PREDICTION:
It's just a habit an

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.031746,0.015723
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...","To the same reason, no attention has been give...",for the same reason no attention has been give...,to the same reason no attention has been given...,0.046512,0.010381
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Rears of these pages will find nothing about t...,readers of these pages will find nothing about...,rears of these pages will find nothing about t...,0.043478,0.012658
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",The part of Chapter 4 has already appeared in ...,part of chapter iv has already appeared in the...,the part of chapter 4 has already appeared in ...,0.163265,0.090909
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,"If Mrs. Shaw had nested at the real, real reas...",if mrs shaw had guessed at the real reason why...,if mrs shaw had nested at the real real reason...,0.128571,0.086842
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","Does he so? said the queen, now comprehending ...",does he so said the queen now comprehending all,does he so said the queen now comprehending all,0.000000,0.000000
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",than if you will promise tonight to keep the o...,then if you will promise to night to keep the ...,than if you will promise tonight to keep the o...,0.176471,0.048193
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","What have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.000000,0.000000
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","that then he rose up, dressed himself hastily,...",then he rose up dressed himself hastily and we...,that then he rose up dressed himself hastily a...,0.076923,0.076923


In [24]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_sad/audio",
    output_csv="evc_jennifer_sad_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [06:53<00:00,  1.03s/it]


Saved CSV:
evc_jennifer_sad_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.1007
Median : 0.0635
Std    : 0.1327
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0502
Median : 0.0297
Std    : 0.0780
Min    : 0.0000
Max    : 0.8773

----- QUALITY -----
Perfect WER: 36.25%
WER <= 0.10: 62.25%
WER <= 0.20: 85.25%
CER <= 0.10: 86.25%

========== TOP 10 WORST WER ==========

5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.5385

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
earhumberg.loukaska
250_142276_000005_000004.wav
WER: 0.9250
CER: 0.8773

GROUND TRUTH:
But he had the same large, soft eyes as his daughter,--eyes which moved slowly and almost grandly round in their orbits, and were well veiled by their transparent white eyelids. Margaret was more like him than like her mother.

PREDICTION:
like hand and like her nother.
5322_7680_000015_000000.wav
WER: 0.8571
CER: 0.2941

GROUND TRUTH:
'Every one...' and Luke swayed his head.

PREDICTION:


,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.047619,0.025157
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,for the same reason no attention has been give...,0.046512,0.034602
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Readers of these pages will find the thing abo...,readers of these pages will find nothing about...,readers of these pages will find the thing abo...,0.173913,0.037975
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",part of chapter four has already appeared in t...,part of chapter iv has already appeared in the...,part of chapter four has already appeared in t...,0.122449,0.065831
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if massa shaw had nested the real reason why m...,if mrs shaw had guessed at the real reason why...,if massa shaw had nested the real reason why m...,0.071429,0.055263
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","""'Does he so?' said the queen, now comprehendi...",does he so said the queen now comprehending all,does he so said the queen now comprehending all,0.000000,0.000000
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",than if he will promise to-night to keep the o...,then if you will promise to night to keep the ...,than if he will promise to night to keep the o...,0.147059,0.060241
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","What have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.074074,0.029630
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [25]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_david/audio",
    output_csv="/content/vc_david_results.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [06:55<00:00,  1.04s/it]


Saved CSV:
/content/vc_david_results.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.0435
Median : 0.0000
Std    : 0.0864
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0168
Median : 0.0000
Std    : 0.0371
Min    : 0.0000
Max    : 0.4615

----- QUALITY -----
Perfect WER: 58.75%
WER <= 0.10: 86.00%
WER <= 0.20: 96.25%
CER <= 0.10: 97.25%

========== TOP 10 WORST WER ==========

5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.4615

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
Hamburg, Dalt Lukaschka,
7800_283492_000058_000000.wav
WER: 0.5000
CER: 0.2344

GROUND TRUTH:
"Horns, Will?" Bluff fired at him; "cows have horns, deer carry antlers!"

PREDICTION:
horns will, they'll have fired at him. How's have horns, dear Kerry Andlers?
6209_34601_000042_000002.wav
WER: 0.4000
CER: 0.0606

GROUND TRUTH:
The savoury odour was perceptible.

PREDICTION:
the savory odor was perceptible.
7800_283493_000070_000000.wav
WER: 0.4000
CER: 0.1000

GROUND TRUTH:
"Go

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.000000,0.000000
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...",for the same reason. No attention has been giv...,for the same reason no attention has been give...,for the same reason no attention has been give...,0.000000,0.000000
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,0.000000,0.000000
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",part of Chapter 4 has already appeared in the ...,part of chapter iv has already appeared in the...,part of chapter 4 has already appeared in the ...,0.061224,0.021944
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,0.014286,0.005263
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","""'Does he so?' said the Queen, now comprehendi...",does he so said the queen now comprehending all,does he so said the queen now comprehending all,0.000000,0.000000
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",than if you will promise tonight to keep the o...,then if you will promise to night to keep the ...,than if you will promise tonight to keep the o...,0.088235,0.012048
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","What have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.037037,0.007407
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [26]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_angry/audio",
    output_csv="evc_david_angry_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [06:59<00:00,  1.05s/it]


Saved CSV:
evc_david_angry_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.1229
Median : 0.0909
Std    : 0.1447
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0665
Median : 0.0399
Std    : 0.0892
Min    : 0.0000
Max    : 0.7317

----- QUALITY -----
Perfect WER: 26.75%
WER <= 0.10: 54.75%
WER <= 0.20: 81.25%
CER <= 0.10: 77.00%

========== TOP 10 WORST WER ==========

5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.4615

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
Air Hamburg. Dat Lukaszka.
5322_7680_000028_000001.wav
WER: 0.8571
CER: 0.3696

GROUND TRUTH:
Bigger than ours?' asked Lukashka good-naturedly.

PREDICTION:
It's bigger than ours. Ask Luke Ashkagan, mate, Trebly.
6209_34600_000004_000003.wav
WER: 0.8000
CER: 0.5714

GROUND TRUTH:
These honours, however, were deserved.

PREDICTION:
Owners, however, or deserved owners,
7800_283478_000011_000001.wav
WER: 0.7647
CER: 0.7317

GROUND TRUTH:
I happened to dodge a ball fired from the 

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.126984,0.147799
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...","For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,0.186047,0.100346
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,His readers of these pages will find nothing a...,readers of these pages will find nothing about...,his readers of these pages will find nothing a...,0.130435,0.082278
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",Part of Chapter 4 has already appeared in the ...,part of chapter iv has already appeared in the...,part of chapter 4 has already appeared in the ...,0.183673,0.156740
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,0.300000,0.197368
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","""'Do as ye so,' said the Queen, now comprehend...",does he so said the queen now comprehending all,do as ye so said the queen now comprehending all,0.333333,0.063830
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",Few will promise to-night to keep the opium-cu...,then if you will promise to night to keep the ...,few will promise to night to keep the opium cu...,0.088235,0.060241
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","Would have I done to thee, that thou should fo...",what have i done to thee that thou shouldst fo...,would have i done to thee that thou should for...,0.074074,0.044444
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [27]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_sad/audio",
    output_csv="evc_david_sad_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [06:55<00:00,  1.04s/it]


Saved CSV:
evc_david_sad_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.1640
Median : 0.1333
Std    : 0.1591
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0978
Median : 0.0675
Std    : 0.1055
Min    : 0.0000
Max    : 0.8788

----- QUALITY -----
Perfect WER: 18.00%
WER <= 0.10: 41.00%
WER <= 0.20: 71.25%
CER <= 0.10: 62.75%

========== TOP 10 WORST WER ==========

5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.5000

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
Hamburg. Dr. Lukaszka.
6209_34601_000156_000001.wav
WER: 0.8571
CER: 0.8788

GROUND TRUTH:
He was answered by a loving growl.

PREDICTION:
out through a loving growl, through well.
250_142276_000006_000004.wav
WER: 0.8571
CER: 0.6061

GROUND TRUTH:
Her out-of-doors life was perfect.

PREDICTION:
for an archonaut of those perfect.
5322_7679_000002_000003.wav
WER: 0.8000
CER: 0.4688

GROUND TRUTH:
The Cossacks received him coldly.

PREDICTION:
Bacassex received from Cobley.
6209_3

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.190476,0.179245
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...",the same reason. No attention has been given t...,for the same reason no attention has been give...,the same reason no attention has been given to...,0.162791,0.138408
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,0.130435,0.056962
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",Chapter 5 of Chapter 4 has already appeared in...,part of chapter iv has already appeared in the...,chapter 5 of chapter 4 has already appeared in...,0.387755,0.257053
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,0.185714,0.150000
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","so, said the Queen, now comprehending all.",does he so said the queen now comprehending all,so said the queen now comprehending all,0.222222,0.170213
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",than if you will promise tonight to keep the o...,then if you will promise to night to keep the ...,than if you will promise tonight to keep the o...,0.264706,0.102410
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","what have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.000000,0.000000
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [28]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_morgan/audio",
    output_csv="/content/vc_morgan_results.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [07:14<00:00,  1.09s/it]


Saved CSV:
/content/vc_morgan_results.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.0542
Median : 0.0000
Std    : 0.2280
Min    : 0.0000
Max    : 4.1176

----- CER -----
Mean   : 0.0280
Median : 0.0000
Std    : 0.1981
Min    : 0.0000
Max    : 3.7857

----- QUALITY -----
Perfect WER: 63.75%
WER <= 0.10: 85.25%
WER <= 0.20: 94.75%
CER <= 0.10: 96.00%

========== TOP 10 WORST WER ==========

6209_34601_000058_000000.wav
WER: 4.1176
CER: 3.7857

GROUND TRUTH:
The man kicked the stool forward and made the little boy sit down, again shoving him by the shoulders; then he pointed with his finger to the porringer which was smoking upon the stove.

PREDICTION:
the man kicked the stool forward and made the little boy sit down again shoving him by the shoulders then he pointed with his finger to the porringer which was smoking upon the stove the Peter tried to prevent the stoner from getting up and up again. Peter told the stoner to stand up, and to be still. the little boy start

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,since my subject is not the splendor of a hist...,since my subject is not the splendor of histor...,since my subject is not the splendor of a hist...,0.015873,0.006289
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,for the same reason no attention has been give...,0.093023,0.079585
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,0.000000,0.000000
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",Part of Chapter 4 has already appeared in the ...,part of chapter iv has already appeared in the...,part of chapter 4 has already appeared in the ...,0.061224,0.021944
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed at the real reason why...,0.014286,0.005263
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","Does he so? said the Queen, now comprehending ...",does he so said the queen now comprehending all,does he so said the queen now comprehending all,0.000000,0.000000
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",then if you will promise to-night to keep the ...,then if you will promise to night to keep the ...,then if you will promise to night to keep the ...,0.000000,0.000000
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","What have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.037037,0.007407
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [29]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_angry/audio",
    output_csv="evc_morgan_angry_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [06:55<00:00,  1.04s/it]


Saved CSV:
evc_morgan_angry_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.1109
Median : 0.0714
Std    : 0.1435
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0528
Median : 0.0284
Std    : 0.0721
Min    : 0.0000
Max    : 0.5238

----- QUALITY -----
Perfect WER: 32.00%
WER <= 0.10: 62.25%
WER <= 0.20: 83.00%
CER <= 0.10: 83.00%

========== TOP 10 WORST WER ==========

5322_7680_000061_000006.wav
WER: 1.0000
CER: 0.4615

GROUND TRUTH:
'Eh, humbug!' thought Lukashka.

PREDICTION:
Air Hamburg, St. Lukaska.
6209_34600_000009_000001.wav
WER: 0.8750
CER: 0.5238

GROUND TRUTH:
Now Melcombe Regis is a parish of Weymouth.

PREDICTION:
Now I'm going to come read jazz. This is a parish of Weymouth.
5322_7679_000002_000003.wav
WER: 0.8000
CER: 0.3750

GROUND TRUTH:
The Cossacks received him coldly.

PREDICTION:
The Cossacks receive Tim Kuzel, please.
5322_7680_000028_000001.wav
WER: 0.7143
CER: 0.3696

GROUND TRUTH:
Bigger than ours?' asked Lukashka good-nature

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.015873,0.009434
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...","For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,0.046512,0.010381
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Readers of these pages will fend nothing about...,readers of these pages will find nothing about...,readers of these pages will fend nothing about...,0.130435,0.056962
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",Part of Chapter 4 has already appeared in the ...,part of chapter iv has already appeared in the...,part of chapter 4 has already appeared in the ...,0.061224,0.021944
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,If Mrs. Shaw had guessed that the real reason ...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed that the real reason w...,0.114286,0.052632
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","That is he so, said the queen, not comprehendi...",does he so said the queen now comprehending all,that is he so said the queen not comprehending...,0.333333,0.148936
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",Then if you will promise tonight to keep the O...,then if you will promise to night to keep the ...,then if you will promise tonight to keep the o...,0.088235,0.018072
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...","What have I done to thee, that thou shouldst f...",what have i done to thee that thou shouldst fo...,what have i done to thee that thou shouldst fo...,0.037037,0.029630
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [30]:
evaluate_asr(
    reference_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/default/all_texts",
    audio_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_sad/audio",
    output_csv="evc_morgan_sad_metrics.csv"
)


Found 400 files


Processing: 100%|██████████| 400/400 [06:48<00:00,  1.02s/it]


Saved CSV:
evc_morgan_sad_metrics.csv


SUMMARY
Files processed: 400

----- WER -----
Mean   : 0.1458
Median : 0.0833
Std    : 0.1988
Min    : 0.0000
Max    : 1.0000

----- CER -----
Mean   : 0.0946
Median : 0.0369
Std    : 0.1628
Min    : 0.0000
Max    : 0.8491

----- QUALITY -----
Perfect WER: 29.25%
WER <= 0.10: 58.75%
WER <= 0.20: 77.75%
CER <= 0.10: 75.50%

========== TOP 10 WORST WER ==========

6209_34599_000009_000000.wav
WER: 1.0000
CER: 0.8333

GROUND TRUTH:
Two or three times the little infant cried.

PREDICTION:
Illich and Pankreich
6209_34600_000007_000002.wav
WER: 1.0000
CER: 0.8333

GROUND TRUTH:
At intervals he knocked at the doors.

PREDICTION:
Thank you.
6209_34599_000018_000002.wav
WER: 1.0000
CER: 0.7826

GROUND TRUTH:
Roofs--dwellings--shelter!

PREDICTION:
CHALTER
5322_7678_000006_000026.wav
WER: 0.9444
CER: 0.8293

GROUND TRUTH:
He thought of God and of the future life as for long he had not thought about them.

PREDICTION:
Out for the bell.
78_369_000069_000010

,filename,ground_truth,prediction,ground_truth_normalized,prediction_normalized,wer,cer
0,250_140277_000004_000000.wav,Since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,since my subject is not the splendor of histor...,0.000000,0.000000
1,250_140277_000004_000001.wav,"For the same reason, no attention has been giv...","For the same reason, no attention has been giv...",for the same reason no attention has been give...,for the same reason no attention has been give...,0.023256,0.020761
2,250_140277_000004_000002.wav,Readers of these pages will find nothing about...,Readers of these pages will find nothing about...,readers of these pages will find nothing about...,readers of these pages will find nothing about...,0.043478,0.031646
3,250_140277_000005_000000.wav,"Part of Chapter IV has already appeared in ""Th...",Part of Chapter 4 has already appeared in the ...,part of chapter iv has already appeared in the...,part of chapter 4 has already appeared in the ...,0.102041,0.065831
4,250_142276_000003_000002.wav,If Mrs. Shaw had guessed at the real reason wh...,if mrs shaw had guessed that the real reason w...,if mrs shaw had guessed at the real reason why...,if mrs shaw had guessed that the real reason w...,0.185714,0.142105
...,...,...,...,...,...,...,...
395,8312_279791_000051_000000.wav,"""Does he so?"" said the queen, now comprehendin...","This is so, said the Queen, now comprehending ...",does he so said the queen now comprehending all,this is so said the queen now comprehending all,0.222222,0.106383
396,8312_279791_000051_000001.wav,"""Then if you will promise to-night to keep the...",then if you will promise to-night to keep the ...,then if you will promise to night to keep the ...,then if you will promise to night to keep the ...,0.029412,0.018072
397,8312_279791_000053_000001.wav,"""What have I done to thee, that thou shouldst ...",that thou should forget me and marry Troutina ...,what have i done to thee that thou shouldst fo...,that thou should forget me and marry troutina ...,0.333333,0.281481
398,8312_279791_000053_000004.wav,"Then he rose up, dressed himself hastily, and ...","Then he rose up, dressed himself hastily, and ...",then he rose up dressed himself hastily and we...,then he rose up dressed himself hastily and we...,0.000000,0.000000


In [ ]:
# @title
from faster_whisper import WhisperModel
from pathlib import Path
from tqdm import tqdm

import librosa
import numpy as np

# =====================================
# Load Whisper once
# =====================================

model = WhisperModel(
    "large-v3",
    device="cuda",
    compute_type="float16"
)

# =====================================
# Check files
# =====================================

def analyze_empty_files(
    folder,
    rms_threshold=1e-4,
    silent_output="silent_files.txt",
    no_speech_output="no_speech_files.txt"
):

    folder = Path(folder)

    wavs = sorted(
        folder.glob("*.wav")
    )

    silent_files = []
    no_speech_files = []

    print(
        f"Checking {len(wavs)} files..."
    )

    for wav in tqdm(
        wavs,
        desc="Analyzing"
    ):

        # -------------------------
        # RMS silence check
        # -------------------------

        try:

            audio, sr = librosa.load(
                wav,
                sr=None,
                mono=True
            )

            rms = np.sqrt(
                np.mean(audio**2)
            )

            if rms < rms_threshold:

                silent_files.append(
                    str(wav)
                )

        except Exception as e:

            print(
                f"Failed RMS check: {wav}"
            )

        # -------------------------
        # Whisper speech check
        # -------------------------

        try:

            segments, _ = model.transcribe(
                str(wav),
                language="en",
                beam_size=5
            )

            text = " ".join(
                seg.text.strip()
                for seg in segments
            ).strip()

            if len(text) == 0:

                no_speech_files.append(
                    str(wav)
                )

        except Exception as e:

            print(
                f"Failed Whisper check: {wav}"
            )

    # =====================================
    # Save lists
    # =====================================

    with open(
        silent_output,
        "w"
    ) as f:

        for path in silent_files:

            f.write(
                path + "\n"
            )

    with open(
        no_speech_output,
        "w"
    ) as f:

        for path in no_speech_files:

            f.write(
                path + "\n"
            )

    # =====================================
    # Summary
    # =====================================

    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)

    print(
        f"Total files: {len(wavs)}"
    )

    print(
        f"Silent files (RMS): "
        f"{len(silent_files)} "
        f"({100*len(silent_files)/len(wavs):.2f}%)"
    )

    print(
        f"No speech files (Whisper): "
        f"{len(no_speech_files)} "
        f"({100*len(no_speech_files)/len(wavs):.2f}%)"
    )

    print()

    print(
        f"Saved silent paths to: "
        f"{silent_output}"
    )

    print(
        f"Saved no-speech paths to: "
        f"{no_speech_output}"
    )

    print("\n========== SILENT FILES ==========\n")

    for p in silent_files:

        print(p)

    print("\n========== NO SPEECH FILES ==========\n")

    for p in no_speech_files:

        print(p)

    return (
        silent_files,
        no_speech_files
    )

In [ ]:
# @title
silent_files, no_speech_files = analyze_empty_files(
    "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_jennifer/audio"
)

In [ ]:
# @title
silent_files, no_speech_files = analyze_empty_files(
    "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_sad/audio"
)

Checking 400 files...


Analyzing: 100%|██████████| 400/400 [07:29<00:00,  1.12s/it]


SUMMARY
Total files: 400
Silent files (RMS): 1 (0.25%)
No speech files (Whisper): 0 (0.00%)

Saved silent paths to: silent_files.txt
Saved no-speech paths to: no_speech_files.txt

========== SILENT FILES ==========

/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_sad/audio/7800_283478_000029_000002.wav

========== NO SPEECH FILES ==========



In [ ]:
# @title
silent_files, no_speech_files = analyze_empty_files(
    "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_angry/audio"
)

Checking 400 files...


Analyzing: 100%|██████████| 400/400 [07:15<00:00,  1.09s/it]


SUMMARY
Total files: 400
Silent files (RMS): 0 (0.00%)
No speech files (Whisper): 0 (0.00%)

Saved silent paths to: silent_files.txt
Saved no-speech paths to: no_speech_files.txt

========== SILENT FILES ==========


========== NO SPEECH FILES ==========



In [ ]:
# @title
silent_files, no_speech_files = analyze_empty_files(
    "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_morgan/audio"
)

Checking 400 files...


Analyzing: 100%|██████████| 400/400 [07:50<00:00,  1.18s/it]


SUMMARY
Total files: 400
Silent files (RMS): 0 (0.00%)
No speech files (Whisper): 0 (0.00%)

Saved silent paths to: silent_files.txt
Saved no-speech paths to: no_speech_files.txt

========== SILENT FILES ==========


========== NO SPEECH FILES ==========



In [ ]:
# @title
silent_files, no_speech_files = analyze_empty_files(
    "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_sad/audio"
)

Checking 400 files...


Analyzing: 100%|██████████| 400/400 [07:18<00:00,  1.10s/it]


SUMMARY
Total files: 400
Silent files (RMS): 4 (1.00%)
No speech files (Whisper): 0 (0.00%)

Saved silent paths to: silent_files.txt
Saved no-speech paths to: no_speech_files.txt

========== SILENT FILES ==========

/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_sad/audio/250_142286_000025_000000.wav
/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_sad/audio/5322_7680_000019_000001.wav
/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_sad/audio/6209_34601_000065_000001.wav
/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_sad/audio/6209_34601_000120_000000.wav

========== NO SPEECH FILES ==========



In [ ]:
# @title
silent_files, no_speech_files = analyze_empty_files(
    "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_angry/audio"
)

Checking 400 files...


Analyzing: 100%|██████████| 400/400 [07:23<00:00,  1.11s/it]


SUMMARY
Total files: 400
Silent files (RMS): 0 (0.00%)
No speech files (Whisper): 0 (0.00%)

Saved silent paths to: silent_files.txt
Saved no-speech paths to: no_speech_files.txt

========== SILENT FILES ==========


========== NO SPEECH FILES ==========



In [ ]:
# @title
silent_files, no_speech_files = analyze_empty_files(
    "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_david/audio"
)

Checking 400 files...


Analyzing: 100%|██████████| 400/400 [07:17<00:00,  1.09s/it]


SUMMARY
Total files: 400
Silent files (RMS): 0 (0.00%)
No speech files (Whisper): 0 (0.00%)

Saved silent paths to: silent_files.txt
Saved no-speech paths to: no_speech_files.txt

========== SILENT FILES ==========


========== NO SPEECH FILES ==========



In [ ]:
# @title
silent_files, no_speech_files = analyze_empty_files(
    "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_sad/audio"
)

Checking 400 files...


Analyzing: 100%|██████████| 400/400 [07:17<00:00,  1.09s/it]


SUMMARY
Total files: 400
Silent files (RMS): 0 (0.00%)
No speech files (Whisper): 0 (0.00%)

Saved silent paths to: silent_files.txt
Saved no-speech paths to: no_speech_files.txt

========== SILENT FILES ==========


========== NO SPEECH FILES ==========



In [ ]:
# @title
silent_files, no_speech_files = analyze_empty_files(
    "/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_angry/audio"
)

Checking 400 files...


Analyzing: 100%|██████████| 400/400 [07:20<00:00,  1.10s/it]


SUMMARY
Total files: 400
Silent files (RMS): 2 (0.50%)
No speech files (Whisper): 0 (0.00%)

Saved silent paths to: silent_files.txt
Saved no-speech paths to: no_speech_files.txt

========== SILENT FILES ==========

/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_angry/audio/7800_283492_000016_000001.wav
/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_angry/audio/7800_283493_000060_000000.wav

========== NO SPEECH FILES ==========



In [2]:
!pip install -q resemblyzer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 8.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 89.4 MB/s eta 0:00:00


In [3]:
from pathlib import Path
from tqdm import tqdm

from resemblyzer import VoiceEncoder, preprocess_wav
from sklearn.metrics.pairwise import cosine_similarity

import pandas as pd
import numpy as np

# =====================================
# Load Resemblyzer once
# =====================================

encoder = VoiceEncoder()

# =====================================
# Embedding
# =====================================

def get_resemblyzer_embedding(audio_path):

    wav = preprocess_wav(audio_path)

    embedding = encoder.embed_utterance(
        wav
    )

    return embedding

# =====================================
# Similarity
# =====================================

def resemblyzer_similarity(
    emb1,
    emb2
):

    score = cosine_similarity(
        emb1.reshape(1, -1),
        emb2.reshape(1, -1)
    )[0][0]

    return float(score)

# =====================================
# Folder Evaluation
# =====================================

def evaluate_resemblyzer_similarity(
    reference_audio,
    generated_dir,
    output_csv=None
):

    generated_dir = Path(
        generated_dir
    )

    generated_files = sorted(
        generated_dir.glob("*.wav")
    )

    print(
        f"Found {len(generated_files)} files"
    )

    # -------------------------
    # Reference embedding ONCE
    # -------------------------

    print(
        "Computing reference embedding..."
    )

    ref_embedding = (
        get_resemblyzer_embedding(
            reference_audio
        )
    )

    results = []

    # -------------------------
    # Compare all files
    # -------------------------

    for audio_file in tqdm(
        generated_files,
        desc="Computing similarity"
    ):

        try:

            gen_embedding = (
                get_resemblyzer_embedding(
                    str(audio_file)
                )
            )

            similarity = (
                resemblyzer_similarity(
                    ref_embedding,
                    gen_embedding
                )
            )

            results.append({

                "filename":
                    audio_file.name,

                "similarity":
                    similarity

            })

        except Exception as e:

            print(
                f"Failed: {audio_file}"
            )

            print(e)

    # -------------------------
    # DataFrame
    # -------------------------

    df = pd.DataFrame(
        results
    )

    # -------------------------
    # Save
    # -------------------------

    if output_csv is not None:

        df.to_csv(
            output_csv,
            index=False
        )

        print(
            f"\nSaved CSV:\n{output_csv}"
        )

    # -------------------------
    # Summary
    # -------------------------

    print("\n" + "=" * 80)
    print("RESEMBLYZER SPEAKER SIMILARITY")
    print("=" * 80)

    print(
        f"Files processed: "
        f"{len(df)}"
    )

    print()

    print(
        f"Mean   : "
        f"{df['similarity'].mean():.4f}"
    )

    print(
        f"Median : "
        f"{df['similarity'].median():.4f}"
    )

    print(
        f"Std    : "
        f"{df['similarity'].std():.4f}"
    )

    print(
        f"Min    : "
        f"{df['similarity'].min():.4f}"
    )

    print(
        f"Max    : "
        f"{df['similarity'].max():.4f}"
    )

    print("\n========== WORST 10 ==========\n")

    print(
        df.sort_values(
            "similarity"
        ).head(10)
    )

    return df

Loaded the voice encoder model on cuda in 0.61 seconds.


In [4]:
df_vc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_jennifer/audio",
    output_csv="jen_netraul_vs_jen_vc_resemblyzer"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:28<00:00, 13.82it/s]



Saved CSV:
jen_netraul_vs_jen_vc_resemblyzer

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.7441
Median : 0.7474
Std    : 0.0298
Min    : 0.6490
Max    : 0.8132

========== WORST 10 ==========

                          filename  similarity
80     5322_7679_000025_000005.wav    0.648983
263  7800_283493_000018_000000.wav    0.654100
43    250_142286_000035_000001.wav    0.658954
295       78_368_000021_000001.wav    0.662125
14    250_142276_000006_000005.wav    0.664394
102    5322_7680_000048_000000.wav    0.664817
197   6209_34601_000114_000001.wav    0.665272
221  7800_283478_000036_000001.wav    0.669374
124   6209_34599_000018_000003.wav    0.670755
13    250_142276_000006_000004.wav    0.671533


In [5]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_sad/audio",
    output_csv="jen_neutral_vs_jen_evc_sad_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:28<00:00, 14.13it/s]


Saved CSV:
jen_neutral_vs_jen_evc_sad_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.7219
Median : 0.7288
Std    : 0.0427
Min    : 0.5501
Max    : 0.8262

========== WORST 10 ==========

                          filename  similarity
106    5322_7680_000057_000000.wav    0.550076
166   6209_34601_000065_000001.wav    0.569982
94     5322_7680_000023_000000.wav    0.571181
330       78_369_000064_000000.wav    0.589574
167   6209_34601_000065_000007.wav    0.595944
280  7800_283493_000070_000000.wav    0.608629
177   6209_34601_000084_000001.wav    0.617744
163   6209_34601_000052_000002.wav    0.619483
102    5322_7680_000048_000000.wav    0.619879
26    250_142286_000006_000000.wav    0.622435


In [7]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_sad_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_sad/audio",
    output_csv="jen_sad_vs_jen_evc_sad_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:20<00:00, 19.47it/s]


Saved CSV:
jen_sad_vs_jen_evc_sad_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8224
Median : 0.8292
Std    : 0.0323
Min    : 0.6790
Max    : 0.8870

========== WORST 10 ==========

                          filename  similarity
335       78_369_000068_000000.wav    0.679013
268  7800_283493_000029_000000.wav    0.692718
204   6209_34601_000156_000001.wav    0.694616
202   6209_34601_000153_000002.wav    0.703777
106    5322_7680_000057_000000.wav    0.707471
13    250_142276_000006_000004.wav    0.709943
315       78_369_000036_000001.wav    0.722428
14    250_142276_000006_000005.wav    0.738966
136   6209_34600_000007_000005.wav    0.739761
31    250_142286_000014_000000.wav    0.748050


In [8]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_angry/audio",
    output_csv="jen_neutral_vs_jen_evc_angry_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:25<00:00, 15.66it/s]


Saved CSV:
jen_neutral_vs_jen_evc_angry_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.6729
Median : 0.6777
Std    : 0.0330
Min    : 0.5147
Max    : 0.7464

========== WORST 10 ==========

                          filename  similarity
365  8312_279790_000039_000001.wav    0.514713
124   6209_34599_000018_000003.wav    0.525513
175   6209_34601_000080_000001.wav    0.556674
177   6209_34601_000084_000001.wav    0.562539
91     5322_7680_000019_000001.wav    0.571355
158   6209_34601_000042_000002.wav    0.581689
152   6209_34601_000008_000001.wav    0.592376
111    5322_7680_000061_000006.wav    0.593273
119   6209_34599_000008_000005.wav    0.593358
235  7800_283492_000010_000001.wav    0.594803


In [9]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/jennifer_angry_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_jennifer_angry/audio",
    output_csv="jen_angry_vs_jen_evc_angry_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:19<00:00, 20.45it/s]


Saved CSV:
jen_angry_vs_jen_evc_angry_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8337
Median : 0.8391
Std    : 0.0284
Min    : 0.7214
Max    : 0.8948

========== WORST 10 ==========

                          filename  similarity
176   6209_34601_000084_000000.wav    0.721434
124   6209_34599_000018_000003.wav    0.724951
123   6209_34599_000018_000002.wav    0.728462
111    5322_7680_000061_000006.wav    0.729797
36    250_142286_000028_000000.wav    0.734474
136   6209_34600_000007_000005.wav    0.745693
177   6209_34601_000084_000001.wav    0.747618
175   6209_34601_000080_000001.wav    0.752415
14    250_142276_000006_000005.wav    0.754339
257  7800_283493_000009_000001.wav    0.756223


In [10]:
df_vc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_david/audio",
    output_csv="david_netraul_vs_david_vc_resemblyzer"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:25<00:00, 15.84it/s]


Saved CSV:
david_netraul_vs_david_vc_resemblyzer

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8490
Median : 0.8534
Std    : 0.0282
Min    : 0.7519
Max    : 0.9074

========== WORST 10 ==========

                          filename  similarity
124   6209_34599_000018_000003.wav    0.751948
295       78_368_000021_000001.wav    0.752645
183   6209_34601_000096_000033.wav    0.755545
14    250_142276_000006_000005.wav    0.761387
221  7800_283478_000036_000001.wav    0.763261
331       78_369_000065_000001.wav    0.771620
222  7800_283478_000037_000001.wav    0.772336
121   6209_34599_000017_000003.wav    0.775593
13    250_142276_000006_000004.wav    0.777012
285       78_368_000008_000000.wav    0.779267


In [11]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_sad/audio",
    output_csv="david_neutral_vs_david_evc_sad_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:27<00:00, 14.58it/s]


Saved CSV:
david_neutral_vs_david_evc_sad_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.7762
Median : 0.7819
Std    : 0.0314
Min    : 0.6614
Max    : 0.8437

========== WORST 10 ==========

                          filename  similarity
204   6209_34601_000156_000001.wav    0.661433
36    250_142286_000028_000000.wav    0.666230
91     5322_7680_000019_000001.wav    0.673259
167   6209_34601_000065_000007.wav    0.676284
244  7800_283492_000026_000000.wav    0.690252
161   6209_34601_000045_000001.wav    0.694879
318       78_369_000043_000008.wav    0.699600
278  7800_283493_000067_000001.wav    0.701850
148   6209_34600_000027_000001.wav    0.703645
116   6209_34599_000003_000000.wav    0.705863


In [12]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_sad_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_sad/audio",
    output_csv="david_sad_vs_david_evc_sad_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:19<00:00, 20.74it/s]


Saved CSV:
david_sad_vs_david_evc_sad_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8333
Median : 0.8369
Std    : 0.0269
Min    : 0.7282
Max    : 0.9020

========== WORST 10 ==========

                          filename  similarity
36    250_142286_000028_000000.wav    0.728208
91     5322_7680_000019_000001.wav    0.743387
161   6209_34601_000045_000001.wav    0.751283
221  7800_283478_000036_000001.wav    0.759477
204   6209_34601_000156_000001.wav    0.762718
58     5322_7678_000006_000014.wav    0.763027
167   6209_34601_000065_000007.wav    0.765874
318       78_369_000043_000008.wav    0.769477
244  7800_283492_000026_000000.wav    0.769896
261  7800_283493_000013_000001.wav    0.772229


In [13]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_angry/audio",
    output_csv="david_neutral_vs_david_evc_angry_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:27<00:00, 14.60it/s]


Saved CSV:
david_neutral_vs_david_evc_angry_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.6364
Median : 0.6408
Std    : 0.0278
Min    : 0.5449
Max    : 0.7011

========== WORST 10 ==========

                          filename  similarity
244  7800_283492_000026_000000.wav    0.544897
88     5322_7680_000014_000000.wav    0.550923
204   6209_34601_000156_000001.wav    0.552157
34    250_142286_000024_000001.wav    0.552444
124   6209_34599_000018_000003.wav    0.559212
327       78_369_000056_000002.wav    0.559310
385  8312_279791_000030_000002.wav    0.563321
207  7800_283478_000011_000001.wav    0.568751
272  7800_283493_000052_000000.wav    0.569106
104    5322_7680_000052_000001.wav    0.573105


In [14]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/david_angry_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_david_angry/audio",
    output_csv="david_angry_vs_david_evc_angry_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:19<00:00, 20.14it/s]


Saved CSV:
david_angry_vs_david_evc_angry_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8623
Median : 0.8661
Std    : 0.0234
Min    : 0.7850
Max    : 0.9106

========== WORST 10 ==========

                          filename  similarity
207  7800_283478_000011_000001.wav    0.785038
36    250_142286_000028_000000.wav    0.785075
244  7800_283492_000026_000000.wav    0.789325
158   6209_34601_000042_000002.wav    0.789596
78     5322_7679_000022_000000.wav    0.796508
348  8312_279790_000012_000000.wav    0.796832
128   6209_34599_000023_000007.wav    0.797457
88     5322_7680_000014_000000.wav    0.802851
150   6209_34600_000029_000006.wav    0.803739
121   6209_34599_000017_000003.wav    0.805916


In [ ]:
# @title
df_vc = evaluate_resemblyzer_similarity(
    reference_audio="/content/chatterbox_TTS.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_jennifer/audio",
    output_csv="chatterbox_vs_jennifer_vc_resemblyzer"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:23<00:00, 17.27it/s]


Saved CSV:
chatterbox_vs_jennifer_vc_resemblyzer

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.6681
Median : 0.6713
Std    : 0.0314
Min    : 0.5654
Max    : 0.7702

========== WORST 10 ==========

                          filename  similarity
167   6209_34601_000065_000007.wav    0.565395
121   6209_34599_000017_000003.wav    0.570225
166   6209_34601_000065_000001.wav    0.577353
106    5322_7680_000057_000000.wav    0.584617
125   6209_34599_000020_000001.wav    0.587666
170   6209_34601_000068_000009.wav    0.588423
199   6209_34601_000134_000000.wav    0.588492
357  8312_279790_000026_000001.wav    0.588655
145   6209_34600_000024_000003.wav    0.589334
104    5322_7680_000052_000001.wav    0.592089


In [ ]:
# @title
df_vc = evaluate_resemblyzer_similarity(
    reference_audio="/content/chatterbox_TTS.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_morgan/audio",
    output_csv="chatterbox_vs_morgan_vc_resemblyzer"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:21<00:00, 19.02it/s]


Saved CSV:
chatterbox_vs_morgan_vc_resemblyzer

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.5568
Median : 0.5600
Std    : 0.0278
Min    : 0.4645
Max    : 0.6211

========== WORST 10 ==========

                         filename  similarity
106   5322_7680_000057_000000.wav    0.464529
163  6209_34601_000052_000002.wav    0.482813
79    5322_7679_000024_000000.wav    0.483720
295      78_368_000021_000001.wav    0.485735
182  6209_34601_000096_000025.wav    0.490624
118  6209_34599_000008_000002.wav    0.491473
91    5322_7680_000019_000001.wav    0.491797
166  6209_34601_000065_000001.wav    0.493032
199  6209_34601_000134_000000.wav    0.493864
177  6209_34601_000084_000001.wav    0.497062


In [ ]:
# @title
df_vc = evaluate_resemblyzer_similarity(
    reference_audio="/content/chatterbox_TTS.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_david/audio",
    output_csv="chatterbox_vs_david_vc_resemblyzer"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:20<00:00, 19.30it/s]


Saved CSV:
chatterbox_vs_david_vc_resemblyzer

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.5371
Median : 0.5396
Std    : 0.0292
Min    : 0.4220
Max    : 0.6041

========== WORST 10 ==========

                          filename  similarity
36    250_142286_000028_000000.wav    0.422010
374  8312_279791_000010_000002.wav    0.438738
299       78_369_000010_000000.wav    0.441642
175   6209_34601_000080_000001.wav    0.462841
199   6209_34601_000134_000000.wav    0.463116
268  7800_283493_000029_000000.wav    0.464709
166   6209_34601_000065_000001.wav    0.465259
198   6209_34601_000120_000000.wav    0.466020
210  7800_283478_000018_000002.wav    0.466200
192   6209_34601_000102_000001.wav    0.469528


In [15]:
df_vc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/vc_morgan/audio",
    output_csv="morgan_netraul_vs_morgan_vc_resemblyzer"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:26<00:00, 14.99it/s]


Saved CSV:
morgan_netraul_vs_morgan_vc_resemblyzer

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8592
Median : 0.8620
Std    : 0.0259
Min    : 0.7743
Max    : 0.9068

========== WORST 10 ==========

                          filename  similarity
150   6209_34600_000029_000006.wav    0.774289
295       78_368_000021_000001.wav    0.785818
199   6209_34601_000134_000000.wav    0.785850
158   6209_34601_000042_000002.wav    0.787911
115    5322_7680_000064_000000.wav    0.788281
125   6209_34599_000020_000001.wav    0.790316
386  8312_279791_000033_000000.wav    0.791615
144   6209_34600_000024_000002.wav    0.791817
374  8312_279791_000010_000002.wav    0.792298
135   6209_34600_000007_000004.wav    0.792814


In [16]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_sad/audio",
    output_csv="morgan_neutral_vs_morgan_evc_sad_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:28<00:00, 13.86it/s]


Saved CSV:
morgan_neutral_vs_morgan_evc_sad_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.7392
Median : 0.7441
Std    : 0.0337
Min    : 0.5102
Max    : 0.8156

========== WORST 10 ==========

                          filename  similarity
133   6209_34600_000007_000002.wav    0.510170
176   6209_34601_000084_000000.wav    0.589729
268  7800_283493_000029_000000.wav    0.601848
339       78_369_000069_000010.wav    0.605898
272  7800_283493_000052_000000.wav    0.625619
150   6209_34600_000029_000006.wav    0.633056
32    250_142286_000015_000003.wav    0.641018
193   6209_34601_000104_000001.wav    0.641672
62     5322_7678_000006_000026.wav    0.651930
58     5322_7678_000006_000014.wav    0.663318


In [17]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_sad_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_sad/audio",
    output_csv="morgan_sad_vs_morgan_evc_sad_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:20<00:00, 19.17it/s]


Saved CSV:
morgan_sad_vs_morgan_evc_sad_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.8242
Median : 0.8309
Std    : 0.0379
Min    : 0.4230
Max    : 0.8908

========== WORST 10 ==========

                          filename  similarity
133   6209_34600_000007_000002.wav    0.422982
176   6209_34601_000084_000000.wav    0.653255
62     5322_7678_000006_000026.wav    0.673623
150   6209_34600_000029_000006.wav    0.682636
115    5322_7680_000064_000000.wav    0.694970
193   6209_34601_000104_000001.wav    0.698693
212  7800_283478_000020_000002.wav    0.724272
123   6209_34599_000018_000002.wav    0.730314
106    5322_7680_000057_000000.wav    0.736924
167   6209_34601_000065_000007.wav    0.741521


In [18]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_angry/audio",
    output_csv="morgan_neutral_vs_morgan_evc_angry_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:27<00:00, 14.33it/s]


Saved CSV:
morgan_neutral_vs_morgan_evc_angry_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.7178
Median : 0.7222
Std    : 0.0258
Min    : 0.5978
Max    : 0.7802

========== WORST 10 ==========

                          filename  similarity
268  7800_283493_000029_000000.wav    0.597790
123   6209_34599_000018_000002.wav    0.606672
186   6209_34601_000096_000057.wav    0.629800
239  7800_283492_000016_000001.wav    0.630882
145   6209_34600_000024_000003.wav    0.646943
166   6209_34601_000065_000001.wav    0.651785
210  7800_283478_000018_000002.wav    0.651971
348  8312_279790_000012_000000.wav    0.652108
307       78_369_000020_000001.wav    0.652625
138   6209_34600_000009_000002.wav    0.653359


In [19]:
df_evc = evaluate_resemblyzer_similarity(
    reference_audio="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/references/morgan_neutral_ref.wav",
    generated_dir="/content/drive/MyDrive/AudioDeepfakesCodes/Data/Speakers_TrainEvalTest/robustness/evc_morgan_angry/audio",
    output_csv="morgan_angry_vs_morgan_evc_angry_resemblyzer.csv"
)

Found 400 files
Computing reference embedding...


Computing similarity: 100%|██████████| 400/400 [00:21<00:00, 18.49it/s]



Saved CSV:
morgan_angry_vs_morgan_evc_angry_resemblyzer.csv

RESEMBLYZER SPEAKER SIMILARITY
Files processed: 400

Mean   : 0.7178
Median : 0.7222
Std    : 0.0258
Min    : 0.5978
Max    : 0.7802

========== WORST 10 ==========

                          filename  similarity
268  7800_283493_000029_000000.wav    0.597790
123   6209_34599_000018_000002.wav    0.606672
186   6209_34601_000096_000057.wav    0.629800
239  7800_283492_000016_000001.wav    0.630882
145   6209_34600_000024_000003.wav    0.646943
166   6209_34601_000065_000001.wav    0.651785
210  7800_283478_000018_000002.wav    0.651971
348  8312_279790_000012_000000.wav    0.652108
307       78_369_000020_000001.wav    0.652625
138   6209_34600_000009_000002.wav    0.653359
